### Load Data

- https://airtable.com/appnbZ1Eod5iNAqGm/tblnzoyM5FGSatizQ/viwtk2bTXcVAPMm1e?blocks=hide

In [ ]:
import pandas as pd
import numpy as np
import json

In [ ]:
file_path = "/Users/emulie/Downloads/script_job_67b7f56b753852fd2a3f35baed75edbc_0.csv" # only for march 01 (one day)
# file_path = "/Users/emulie/Downloads/bq-results-20260325-195823-1774468728937.csv" # for Feb 1st to March 1st


In [ ]:
SOUNDS_IDS_JSON_PATH = "../data/sounds_ids.json"
with open(SOUNDS_IDS_JSON_PATH) as f:
    sounds_ids = json.load(f)

In [ ]:
df = pd.read_csv(file_path)

import json

df['sounds'] = df['sounds'].apply(json.loads)
df['sounds_volume'] = df['sounds_volume'].apply(json.loads)

df_sounds = df.explode(['sounds', 'sounds_volume'], ignore_index=True)
df_sounds['sounds_volume'] = df_sounds['sounds_volume'].astype(float)
df_sounds.head(10)

df_listening = df.copy()

In [ ]:
df.head()

In [ ]:
# reading with duckdb
import duckdb

con = duckdb.connect()
results_df = con.execute(f"SELECT * FROM read_csv('{file_path}')").fetch_df()
numpy_array = results_df.to_numpy()

### Questions

EDA
- how many sounds? Airtable says 298
- Length of mix? Max? Average?
- Average Sound Volume for each sounds? Is it necessary to have a volume predictor?

Remapping:
- are there sounds to be remapped? Should be remove prefixes (eg. `ambience.<>`)



In [ ]:
# Q1: How many unique sounds? (Airtable says 298)
n_unique = df_sounds['sounds'].nunique()
print(f"Unique sounds in data: {n_unique} (Airtable: 298)")

In [ ]:
# Q: what's the average volume and variance
df_sounds_meta = df_sounds.groupby('sounds')['sounds_volume'].agg(list).reset_index()
df_sounds_meta['sounds_volume'] = df_sounds_meta['sounds_volume'].apply(sorted)
df_sounds_meta['volume_average'] = df_sounds_meta['sounds_volume'].apply(np.mean)
df_sounds_meta['volume_variance'] = df_sounds_meta['sounds_volume'].apply(np.var)
df_sounds_meta['volume_count'] = df_sounds_meta['sounds_volume'].apply(lambda x: len(x))

print(f"Max Volume Count: {df_sounds_meta['volume_count'].max()}")
print(f"Max Volume Variance: {df_sounds_meta['volume_variance'].max()}")

df_sounds_meta

In [ ]:
# Q2: Mix length — how many sounds per session?
mix_lengths = df['sounds'].apply(len)
print(f"Average mix length: {mix_lengths.mean():.2f}")
print(f"Max mix length:     {mix_lengths.max()}")
print(f"Min mix length:     {mix_lengths.min()}")
mix_lengths.value_counts().sort_index().rename("sessions").to_frame()

In [ ]:
# Q3: Average volume per sound (sorted descending)
avg_volume = (
    df_sounds
    .groupby('sounds')['sounds_volume']
    .agg(avg_volume='mean', n_plays='count')
    .sort_values('avg_volume', ascending=False)
    .round(3)
)
avg_volume

In [ ]:
unique_sounds = df_sounds['sounds'].unique().tolist()
unique_sounds.sort()

In [ ]:
# prefix
set([name.split('.')[0] for name in unique_sounds if '.' in name])

In [ ]:
# remove prefix to see which sounds can be re-mapped
len(list(set([name.split('.')[-1] for name in unique_sounds])))

### Data Cleaning


- remove mixes with only one sounds
- normalize for popular sounds
- remap sounds (if needed)

In [ ]:
allowed_prefixes = ['ambience', 'asmr', 'binaural', 'isochronic', 'music', 'solfeggio']


In [ ]:
all_sounds = sorted(df_sounds['sounds'].unique())
# list(all_sounds)

In [ ]:
len([s for s in all_sounds if '.' in s ])

In [ ]:
len(sounds_ids)

### Mix × Sounds Matrix

In [ ]:
# TODO: remove sound not in sounds_id from 'sounds'


In [ ]:
all_sounds = set(sounds_ids)
df_clean = df_listening.copy()
df_clean['sounds'] = df_listening['sounds'].apply(lambda x: [s for s in x if s in all_sounds])

print((df_clean['sounds'] == df_listening['sounds']).eq(False).any())

# changed = df_clean['sounds'].apply(len) != df_listening['sounds'].apply(len)
# print(changed.any())
# df_clean['sounds'].explode(['sounds']).unique()

df_clean = df_clean[df_clean['sounds'].apply(len) > 2]

In [ ]:
from scipy.sparse import csr_matrix

df = df_clean
# df = df_listening
# 1. Mix signature: order-invariant sorted string, e.g. "ambience.ocean|ambience.rain"
df['mix_id'] = df['sounds'].apply(lambda s: '|'.join(sorted(s)))

# 2. Count how many sessions used each mix
mix_counts = df.groupby('mix_id').size().reset_index(name='play_count')
print(f"Unique mixes: {len(mix_counts):,}")
mix_counts.head()

In [ ]:
# 3. Explode mix → individual sounds, carry play_count as the interaction weight
df_mix = (
    df[['mix_id', 'sounds']]
    .merge(mix_counts, on='mix_id')
    .explode('sounds', ignore_index=True)
)

# 4. Encode mix_id and sound to integer indices
df_mix['mix_idx'] = df_mix['mix_id'].astype('category').cat.codes
mix_labels = df_mix['mix_id'].astype('category').cat.categories  # int → mix_id

sound_cat = pd.CategoricalDtype(categories=sounds_ids, ordered=False)
df_mix['sound_idx'] = df_mix['sounds'].astype(sound_cat).cat.codes
sound_labels = sounds_ids  # aligned with sounds_ids order

n_mixes  = len(mix_labels)
n_sounds = len(sounds_ids)
print(f"Matrix shape: {n_mixes:,} mixes × {n_sounds} sounds")


# 5. Build sparse mix × sound matrix
#    Value = play_count of the mix (how many sessions used it)
mix_item = csr_matrix(
    (df_mix['play_count'].astype('float32'),
     (df_mix['mix_idx'], df_mix['sound_idx'])),
    shape=(n_mixes, n_sounds)
)
mix_item

### Training

In [ ]:
from implicit.nearest_neighbours import bm25_weight
from implicit.als import AlternatingLeastSquares

# BM25 weighting: reduces dominance of very popular mixes
# implicit expects (item × user), so we transpose: (sound × mix)
mix_item_weighted = bm25_weight(mix_item.T, K1=100, B=0.8).T.tocsr()

model = AlternatingLeastSquares(factors=64, regularization=0.1, alpha=0.05)
model.fit(mix_item_weighted)

In [ ]:
# Quick sanity check: recommend sounds for the first mix in the matrix
mix_idx = 150
ids, scores = model.recommend(mix_idx, mix_item_weighted[mix_idx], N=10, filter_already_liked_items=True)

current_mix = mix_labels[mix_idx].split('|')
print(f"Current mix: {current_mix}\n")
pd.DataFrame({"recommended_sound": np.array(sound_labels)[ids], "score": scores.round(4)})


In [ ]:
# ---- inference with any mixes
custom_mix = ['ambience.airconditioner', 'ambience.ocean', 'ambience.pinknoise']
sound_to_idx = {s: i for i, s in enumerate(sound_labels)}
indices = [sound_to_idx[s] for s in current_mix if s in sound_to_idx]

# build sparse vector
from scipy.sparse import csr_matrix
import numpy as np

data = np.ones(len(indices))  # or weights if you have them
row = np.zeros(len(indices))  # single "user"
col = indices

custom_vector = csr_matrix((data, (row, col)), shape=(1, len(sound_labels)))

ids, scores = model.recommend(
    userid=0,  # dummy (ignored)
    user_items=custom_vector,
    N=10,
    filter_already_liked_items=True
)

print(custom_mix)
pd.DataFrame({
    "recommended_sound": np.array(sound_labels)[ids],
    "score": scores.round(4)
})

In [ ]:
# np.array(sound_labels)[ids]
# scores.round(4)

### Converting to PyTorch

In [ ]:
model.item_factors # (863, 64)
model.user_factors # (12952, 64)
model.alpha # 2.0
model.regularization # 0.05

In [ ]:
import numpy as np
import torch
import torch.nn as nn

item_factors = model.item_factors  # [n_sounds, 64]
user_factors = model.user_factors  # [n_mixes,  64]
# mix_emb = (S.T @ C @ S + λI)^-1 @ S.T @ C @ r
# S = sound_embeddings — [num_sounds, D]
# C = diagonal confidence matrix for this mix — diag(1 + alpha * r)
# r = the mix's sound vector — [num_sounds] binary
# λ = regularization

# Fold-in matrix: lets us derive a latent vector for an unseen mix
# given its interaction vector (sound volumes)
# score = mix_emb @ sound_emb.T + mix_emb @ (YtY_reg)^-1 @ sound_emb.T
YtY = item_factors.T @ item_factors           # [64, 64]
reg = np.eye(64) * model.regularization
foldin_matrix = np.linalg.solve(YtY + reg, item_factors.T)  # [64, n_sounds]


class ALSRecommender(nn.Module):
    def __init__(self, item_factors: np.ndarray, foldin_matrix: np.ndarray):
        super().__init__()
        self.register_buffer("item_factors",  torch.tensor(item_factors,  dtype=torch.float32))
        self.register_buffer("foldin_matrix", torch.tensor(foldin_matrix, dtype=torch.float32))

    def forward(self, interaction_vector: torch.Tensor) -> torch.Tensor:
        """
        Args:
            interaction_vector: [n_sounds] float tensor — volume at index i if sound i
                                is in the current mix, else 0.0
        Returns:
            scores: [n_sounds] float tensor — higher = better recommendation
        """
        user_vec = self.foldin_matrix @ interaction_vector  # [64]
        scores   = self.item_factors   @ user_vec           # [n_sounds]
        return scores


als_model = ALSRecommender(item_factors, foldin_matrix)
als_model.eval()

In [ ]:
# fold-in
import torch

NUM_SOUNDS = 501
mix = torch.tensor([1]*NUM_SOUNDS)

S = model.item_factors
r = model.user_factors
alpha = model.alpha
reg = model.regularization

c = 1 + alpha * mix
StCS = S.T @ (c.unsqueeze(1) * S)
rhs = S.T @ (c * r)
reg_matrix = reg * torch.eye(S.shape[1])
query = torch.linalg.solve(StCS + reg_matrix, rhs)

In [ ]:
# Verify PyTorch output matches NumPy fold-in
r = np.zeros(n_sounds, dtype=np.float32)
r[0] = 0.8  # first sound at vol 0.8

expected = item_factors @ (foldin_matrix @ r)

with torch.no_grad():
    pt_scores = als_model(torch.tensor(r)).numpy()

np.testing.assert_allclose(pt_scores, expected, rtol=1e-4)
print("✓ PyTorch output matches NumPy")

### Export to LiteRT using TensorFlow

In [ ]:
import tensorflow as tf

inp = tf.keras.Input(shape=(n_sounds,), name="interaction_vector")
user_vec = tf.keras.layers.Dense(
    64, use_bias=False, name="foldin",
    kernel_initializer=tf.constant_initializer(foldin_matrix.T)
)(inp)
scores_out = tf.keras.layers.Dense(
    n_sounds, use_bias=False, name="item_scores",
    kernel_initializer=tf.constant_initializer(item_factors)
)(user_vec)

keras_model = tf.keras.Model(inputs=inp, outputs=scores_out)
keras_model.trainable = False

converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

output_path = "als_recommender.tflite"
with open(output_path, "wb") as f:
    f.write(tflite_model)

print(f"Saved {output_path} ({len(tflite_model) / 1024:.1f} KB)")

### Convert model to coreml

In [ ]:
# import coremltools as ct 

# # sample_input = np.zeros(n_sounds, dtype=np.float32)
# sample_input = torch.zeros(n_sounds, dtype=np.float32)
# traced_model = torch.jit.trace(als_model, sample_input)

# # mlmodel = ct.convert(traced_model, inputs=[sample_input])


### Verify TFLite output matches PyTorch

In [ ]:
interpreter = tf.lite.Interpreter(model_path=output_path)
interpreter.allocate_tensors()

inp_det = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]

interpreter.set_tensor(inp_det['index'], r[np.newaxis, :])  # add batch dim
interpreter.invoke()
tflite_scores = interpreter.get_tensor(out_det['index'])[0]

# np.testing.assert_allclose(tflite_scores, pt_scores, rtol=1e-2)  # looser: quantization noise
np.testing.assert_allclose(tflite_scores, pt_scores, atol=1e-6)  # looser: quantization noise
print("✓ TFLite output matches PyTorch")

### Inference Example

In [ ]:
current_mix = {
    "ambience.rain":  0.8,
    "ambience.ocean": 0.5,
}

current_mix = {
    "ambience.rain":  1.0,
    "ambience.ocean": 1.0,
}

# Build interaction vector: volume at the sound's index, 0 elsewhere
r_infer = np.zeros(n_sounds, dtype=np.float32) # n_sounds = 863
for sound, vol in current_mix.items():
    if sound in sound_labels:
        r_infer[sound_labels.get_loc(sound)] = vol

# Run model
with torch.no_grad():
    scores = als_model(torch.tensor(r_infer)).numpy()

# Mask out sounds already in the mix, then return top 5
already_playing = set(current_mix.keys())
results = (
    pd.Series(scores, index=sound_labels)
    .drop(labels=[s for s in already_playing if s in sound_labels])
    .sort_values(ascending=False)
    .head(5)
    .rename("score")
    .reset_index()
    .rename(columns={"index": "recommended_sound"})
)
results

### Evaluation

In [ ]:
# --- Leave-one-out test set ---
# For each mix with ≥2 sounds: hold out 1 random sound, use the rest as query
np.random.seed(42)

def split_mix(sounds):
    i = np.random.randint(len(sounds))
    return sounds[:i] + sounds[i+1:], sounds[i]  # query, hold_out

test_df = (
    df[df['sounds'].apply(len) >= 2]['sounds']
    .apply(lambda s: pd.Series(split_mix(s), index=['query', 'hold_out']))
    .reset_index(drop=True)
    .sample(min(2000, len(df)), random_state=42)  # cap for speed; use all rows with full data
    .reset_index(drop=True)
)
print(f"Test mixes: {len(test_df):,}")
test_df.head(3)

In [ ]:
K = 10

def recommend(query_sounds, k=K):
    r = np.zeros(n_sounds, dtype=np.float32)
    for s in query_sounds:
        if s in sound_labels:
            r[sound_labels.get_loc(s)] = 1.0
    with torch.no_grad():
        scores = als_model(torch.tensor(r)).numpy()
    # mask out sounds already in the query
    for s in query_sounds:
        if s in sound_labels:
            scores[sound_labels.get_loc(s)] = -np.inf
    top_k = np.argsort(scores)[::-1][:k]
    return list(sound_labels[top_k])

test_df['recs'] = test_df['query'].apply(recommend)
test_df.head(3)

In [ ]:
# --- Ranking Metrics ---
def hit(recs, hold_out):
    return int(hold_out in recs)

def reciprocal_rank(recs, hold_out):
    return 1.0 / (recs.index(hold_out) + 1) if hold_out in recs else 0.0

def ndcg(recs, hold_out):
    if hold_out in recs:
        rank = recs.index(hold_out) + 1
        return np.log2(2) / np.log2(rank + 1)
    return 0.0

test_df['hit']  = test_df.apply(lambda r: hit(r['recs'], r['hold_out']), axis=1)
test_df['rr']   = test_df.apply(lambda r: reciprocal_rank(r['recs'], r['hold_out']), axis=1)
test_df['ndcg'] = test_df.apply(lambda r: ndcg(r['recs'], r['hold_out']), axis=1)

print(f"Hit Rate@{K}: {test_df['hit'].mean():.4f}")
print(f"MRR@{K}:      {test_df['rr'].mean():.4f}")
print(f"NDCG@{K}:     {test_df['ndcg'].mean():.4f}")

In [ ]:
# --- Diversity (Intra-list) ---
# Average pairwise cosine distance between item_factors of recommended sounds.
# Higher = more varied recommendations.
from sklearn.metrics.pairwise import cosine_distances

def intra_list_diversity(recs):
    idxs = [sound_labels.get_loc(s) for s in recs if s in sound_labels]
    if len(idxs) < 2:
        return 0.0
    vecs = item_factors[idxs]
    dists = cosine_distances(vecs)
    n = len(idxs)
    return dists[np.triu_indices(n, k=1)].mean()

test_df['diversity'] = test_df['recs'].apply(intra_list_diversity)
print(f"Avg Intra-list Diversity@{K}: {test_df['diversity'].mean():.4f}  (range 0–2, higher = more diverse)")

In [ ]:
# --- Novelty ---
# Popularity = fraction of mixes containing the sound.
# Novelty of a list = mean(-log2(popularity)) — rewards recommending niche sounds.
sound_pop = df['sounds'].explode().value_counts() / len(df)

def novelty(recs):
    pops = [sound_pop.get(s, 1e-6) for s in recs]
    return np.mean(-np.log2(pops))

test_df['novelty'] = test_df['recs'].apply(novelty)
print(f"Avg Novelty@{K}: {test_df['novelty'].mean():.4f}  (higher = less popular sounds recommended)")

In [ ]:
# --- Serendipity ---
# Relevant (= hold_out is in recs) AND unexpected (= not among globally top-K popular sounds).
# Rewards recommendations that are both correct and non-obvious.
popular_baseline = set(sound_pop.nlargest(K).index)

def serendipity(recs, hold_out):
    return int(hold_out in recs and hold_out not in popular_baseline) / K

test_df['serendipity'] = test_df.apply(lambda r: serendipity(r['recs'], r['hold_out']), axis=1)
print(f"Avg Serendipity@{K}: {test_df['serendipity'].mean():.4f}")
print(f"(Popular baseline sounds: {sorted(popular_baseline)})")

### Sound Stats

In [ ]:
r_infer

In [ ]:
scores